# pysnapgene quickstart

Read and write SnapGene `.dna` files from Python, keeping the colours, feature types and
translation flags that other tools drop.

This notebook is self-contained: it builds a small plasmid, writes it, reads it back and
shows what survived. The last section shows how to point it at one of your own files.

```bash
conda env create -f ../environment.yml
conda activate pysnapgene
pip install -e "..[dev]"
```

---
## 1. Build a plasmid in Python

In [1]:
from pathlib import Path

from Bio.Seq import Seq
from Bio.SeqFeature import SeqFeature, SimpleLocation
from Bio.SeqRecord import SeqRecord

import pysnapgene
from pysnapgene import Q_COLOR, Q_TRANSLATED, read_snapgene, write_snapgene

print("pysnapgene", pysnapgene.__version__)

# a toy construct: promoter - tag - linker - ORF - His6 - stop
PROMOTER = "TTGACAATTAATCATCGGCTCGTATAATG"
TAG = "ATGGTGAGCAAGGGCGAGGAGCTGTTCACC"          # 30 nt, 10 codons
LINKER = "GGCGGATCCGGAGGTTCTGGC"                  # 21 nt, 7 codons
ORF = "GAATATAAACTGGTGGTGGTGGGCGCGGATGGCGTGGGCAAAAGC"  # 45 nt, 15 codons
HIS = "CATCATCACCACCACCAC"                        # 18 nt, 6 codons
STOP = "TGA"
FILLER = "GCGGCCGCTTAATTAAGGCGCGCCATCGATACGCGTCCTGCAGG"

sequence = PROMOTER + TAG + LINKER + ORF + HIS + STOP + FILLER
record = SeqRecord(
    Seq(sequence),
    id="pDemo001",
    name="pDemo001",
    description="pDemo001 pysnapgene quickstart construct",
)
record.annotations["molecule_type"] = "DNA"
record.annotations["topology"] = "circular"

offset = 0
blocks = [
    ("T7_promoter", PROMOTER, "promoter", "#ffffff"),
    ("mNeonGreen_frag", TAG, "CDS", "#75c6a9"),
    ("GS_linker", LINKER, "CDS", "#d6d6d6"),
    ("KRAS_frag", ORF, "CDS", "#f58a5e"),
    ("His6", HIS, "CDS", "#c7b0e3"),
    ("stop", STOP, "misc_feature", "#ff0000"),
]
for label, block, ftype, colour in blocks:
    record.features.append(
        SeqFeature(
            SimpleLocation(offset, offset + len(block), strand=1),
            type=ftype,
            qualifiers={"label": [label], Q_COLOR: [colour]},
        )
    )
    offset += len(block)

# a sequencing primer, which belongs in SnapGene's Primers panel rather than the features
record.features.append(
    SeqFeature(
        SimpleLocation(5, 28, strand=1),
        type="primer_bind",
        qualifiers={"label": ["pDemo_fwd"]},
    )
)

print(f"{len(record.seq)} bp, {len(record.features)} features")

pysnapgene 0.1.0
190 bp, 7 features


---
## 2. Write it, read it back

`write_snapgene` produces a file SnapGene opens directly. Reading it back gives an ordinary
`SeqRecord` with the SnapGene display properties attached as qualifiers.

In [2]:
out = Path("pDemo001.dna")
write_snapgene(record, out)
print(f"wrote {out} ({out.stat().st_size} bytes)")

back = read_snapgene(out)
print(f"read back: {len(back.seq)} bp, topology={back.annotations['topology']}, "
      f"double stranded={back.annotations['double_stranded']}")
print(f"sequence identical: {str(back.seq) == str(record.seq)}")

print(f"\n{'feature':20} {'type':14} {'colour':9} {'translated':11} protein")
for feature in back.features:
    label = feature.qualifiers.get("label", ["?"])[0]
    colour = feature.qualifiers.get(Q_COLOR, ["-"])[0]
    translated = "yes" if feature.qualifiers.get(Q_TRANSLATED) else "-"
    protein = feature.qualifiers.get("translation", [""])[0]
    print(f"{label:20} {feature.type:14} {colour:9} {translated:11} {protein}")

wrote pDemo001.dna (3051 bytes)
read back: 190 bp, topology=circular, double stranded=True
sequence identical: True

feature              type           colour    translated  protein
T7_promoter          promoter       #ffffff   -           
mNeonGreen_frag      CDS            #75c6a9   yes         MVSKGEELFT
GS_linker            CDS            #d6d6d6   yes         GGSGGSG
KRAS_frag            CDS            #f58a5e   yes         EYKLVVVGADGVGKS
His6                 CDS            #c7b0e3   yes         HHHHHH
stop                 misc_feature   #ff0000   -           
pDemo_fwd            primer_bind    -         -           


Colours came back, the CDS features are marked translated, and the primer is a
`primer_bind` feature because it was stored in the Primers packet rather than as an
annotation.

---
## 3. Translation is only claimed when the span really is an ORF

Feature boundaries copied from another map are often out of frame. Marking those as CDS
makes SnapGene show a nonsense protein, so the writer checks first: a whole number of
codons, and no internal stop. Anything failing that keeps its colour and position but is
written as `misc_feature`.

In [3]:
import copy

probe = copy.deepcopy(record)
probe.features.append(
    SeqFeature(                       # 20 nt: not a whole number of codons
        SimpleLocation(0, 20, strand=1),
        type="CDS",
        qualifiers={"label": ["ragged_cds"], Q_COLOR: ["#ff0000"]},
    )
)
orf_start = len(PROMOTER)
past_stop = len(PROMOTER + TAG + LINKER + ORF + HIS + STOP) + 6
probe.features.append(
    SeqFeature(                       # runs straight through the stop codon
        SimpleLocation(orf_start, past_stop, strand=1),
        type="CDS",
        qualifiers={"label": ["readthrough"], Q_COLOR: ["#ff0000"]},
    )
)
write_snapgene(probe, "probe.dna")
checked = read_snapgene("probe.dna")

for label in ("ragged_cds", "readthrough", "KRAS_frag"):
    f = [x for x in checked.features if x.qualifiers.get("label", [""])[0] == label][0]
    print(f"{label:14} type={f.type:14} translated="
          f"{'yes' if f.qualifiers.get(Q_TRANSLATED) else 'no':3} "
          f"colour={f.qualifiers.get(Q_COLOR, ['-'])[0]}")

ragged_cds     type=misc_feature   translated=no  colour=#ff0000
readthrough    type=misc_feature   translated=no  colour=#ff0000
KRAS_frag      type=CDS            translated=yes colour=#f58a5e


---
## 4. Recolour an existing map

Because colours are plain qualifiers, restyling a plasmid is a dict update.

In [4]:
PALETTE = {
    "promoter": "#ffffff",
    "CDS": "#b1ff67",
    "misc_feature": "#ffef86",
}
restyled = read_snapgene("pDemo001.dna")
for feature in restyled.features:
    if feature.type in PALETTE:
        feature.qualifiers[Q_COLOR] = [PALETTE[feature.type]]

write_snapgene(restyled, "pDemo001_restyled.dna")
again = read_snapgene("pDemo001_restyled.dna")
for feature in again.features:
    print(f"{feature.qualifiers.get('label', ['?'])[0]:20} "
          f"{feature.type:14} {feature.qualifiers.get(Q_COLOR, ['-'])[0]}")

T7_promoter          promoter       #ffffff
mNeonGreen_frag      CDS            #b1ff67
GS_linker            CDS            #b1ff67
KRAS_frag            CDS            #b1ff67
His6                 CDS            #b1ff67
stop                 misc_feature   #ffef86
pDemo_fwd            primer_bind    -


---
## 5. Inspect the raw packets

A `.dna` file is a flat run of TLV packets. The packet layer is public, which is handy when
something does not look right in SnapGene.

In [5]:
from pysnapgene import iter_packets

for packet in iter_packets(Path("pDemo001.dna").read_bytes()):
    preview = packet.data[:48]
    print(f"0x{packet.type:02x}  {packet.name:28} {len(packet):>7} bytes  {preview!r}")

0x09  cookie                            14 bytes  b'SnapGene\x00\x01\x00\x0f\x00\x14'
0x00  DNA                              191 bytes  b'\x1fTTGACAATTAATCATCGGCTCGTATAATGATGGTGAGCAAGGGCGAG'
0x0a  features                        2146 bytes  b'<?xml version="1.0"?><Features nextValidID="6"><'
0x05  primers                          417 bytes  b'<?xml version="1.0"?><Primers nextValidID="1"><H'
0x06  notes                            258 bytes  b'<Notes><UUID>8377ef57-aa32-436d-aa42-925c6aba936'


In [6]:
# the flag byte is the one that catches people out
from pysnapgene import (
    FLAG_CIRCULAR,
    FLAG_DAM_METHYLATED,
    FLAG_DCM_METHYLATED,
    FLAG_DOUBLE_STRANDED,
    FLAG_ECOKI_METHYLATED,
)

flags = back.annotations["snapgene_flags"]
bits = [
    ("circular", FLAG_CIRCULAR),
    ("double stranded", FLAG_DOUBLE_STRANDED),
    ("Dam methylated", FLAG_DAM_METHYLATED),
    ("Dcm methylated", FLAG_DCM_METHYLATED),
    ("EcoKI methylated", FLAG_ECOKI_METHYLATED),
]
print(f"DNA flag byte: 0x{flags:02x}")
for name, bit in bits:
    print(f"   {name:18} {'yes' if flags & bit else 'no'}")
print("\nLeave the double-stranded bit clear and SnapGene shows the molecule as ssDNA.")

DNA flag byte: 0x1f
   circular           yes
   double stranded    yes
   Dam methylated     yes
   Dcm methylated     yes
   EcoKI methylated   yes

Leave the double-stranded bit clear and SnapGene shows the molecule as ssDNA.


---
## 6. Round-trip one of your own files

Point `SOURCE` at any `.dna` SnapGene wrote. Unrecognised packets — enzyme sets,
alignments, edit history — are carried through untouched, so nothing is lost.

In [7]:
SOURCE = None   # e.g. Path.home() / "Desktop" / "my_plasmid.dna"

if SOURCE and Path(SOURCE).exists():
    original = read_snapgene(SOURCE)
    write_snapgene(original, "roundtrip.dna")
    copy_back = read_snapgene("roundtrip.dna")

    def key(f):
        return (int(f.location.start), int(f.location.end), f.location.strand,
                f.qualifiers.get("label", ["?"])[0],
                f.qualifiers.get(Q_COLOR, ["?"])[0])

    kept = {key(f) for f in original.features
            if not (f.type == "primer_bind"
                    and int(f.location.end) - int(f.location.start) < 15)}
    print(f"{len(original.seq)} bp")
    print(f"sequence identical        : {str(original.seq) == str(copy_back.seq)}")
    print(f"features + colours identical: {kept == {key(f) for f in copy_back.features}}")
    print(f"unknown packets preserved : "
          f"{len(copy_back.annotations['snapgene_unknown_packets'])}")
else:
    print("Set SOURCE to a .dna file to run this section.")

Set SOURCE to a .dna file to run this section.


In [8]:
# tidy up the files this notebook created
for name in ("pDemo001.dna", "pDemo001_restyled.dna", "probe.dna", "roundtrip.dna"):
    Path(name).unlink(missing_ok=True)
print("cleaned up")

cleaned up
